<h2> MongoDB Part

In [1]:
import pymongo
from pymongo import MongoClient
import pandas as pd
client = MongoClient('localhost', 27017)
db = client.example

import pandas as pd
import pymysql
import getpass # May need to import as `from getpass import getpass`
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
import sqlite3

In [2]:
airlines = pd.read_csv("airlines.csv")
airports = pd.read_csv("airports.csv")
flights = pd.read_csv("flights.csv", low_memory=False)

In [3]:
flights_airlines = pd.merge(airlines, flights, left_on='IATA_CODE', right_on = 'AIRLINE', how = 'inner' )
flights_airlines
#a.IATA_CODE = f.AIRLINE

,IATA_CODE,AIRLINE_x,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE_y,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,UA,United Air Lines Inc.,2015,1,1,4,UA,1197,N78448,SFO,...,619.0,-7.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,UA,United Air Lines Inc.,2015,1,1,4,UA,1545,N76517,LAX,...,607.0,-11.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,UA,United Air Lines Inc.,2015,1,1,4,UA,1528,N76519,SJU,...,458.0,-11.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,UA,United Air Lines Inc.,2015,1,1,4,UA,1162,N37293,BQN,...,605.0,6.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,UA,United Air Lines Inc.,2015,1,1,4,UA,1500,N30401,ORD,...,816.0,11.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5819074,VX,Virgin America,2015,12,31,4,VX,769,N622VA,LGA,...,2154.0,-6.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
5819075,VX,Virgin America,2015,12,31,4,VX,357,N284VA,BOS,...,2204.0,-46.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
5819076,VX,Virgin America,2015,12,31,4,VX,1916,N853VA,SFO,...,2052.0,-18.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
5819077,VX,Virgin America,2015,12,31,4,VX,490,N840VA,LAX,...,2044.0,-11.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
sample = flights_airlines.sample(10000, random_state=1)
sample_dict = sample.to_dict('records')
flights_airlines_collection = db.flights_airlines
result0 = flights_airlines_collection.insert_many(sample_dict)

In [5]:
result = flights_airlines_collection.find_one({'YEAR': 2015})
print(result)

{'_id': ObjectId('6a1931516d08ca1732498ef9'), 'IATA_CODE': 'UA', 'AIRLINE_x': 'United Air Lines Inc.', 'YEAR': 2015, 'MONTH': 1, 'DAY': 1, 'DAY_OF_WEEK': 4, 'AIRLINE_y': 'UA', 'FLIGHT_NUMBER': 1197, 'TAIL_NUMBER': 'N78448', 'ORIGIN_AIRPORT': 'SFO', 'DESTINATION_AIRPORT': 'IAH', 'SCHEDULED_DEPARTURE': 48, 'DEPARTURE_TIME': 42.0, 'DEPARTURE_DELAY': -6.0, 'TAXI_OUT': 11.0, 'WHEELS_OFF': 53.0, 'SCHEDULED_TIME': 218.0, 'ELAPSED_TIME': 217.0, 'AIR_TIME': 199.0, 'DISTANCE': 1635, 'WHEELS_ON': 612.0, 'TAXI_IN': 7.0, 'SCHEDULED_ARRIVAL': 626, 'ARRIVAL_TIME': 619.0, 'ARRIVAL_DELAY': -7.0, 'DIVERTED': 0, 'CANCELLED': 0, 'CANCELLATION_REASON': nan, 'AIR_SYSTEM_DELAY': nan, 'SECURITY_DELAY': nan, 'AIRLINE_DELAY': nan, 'LATE_AIRCRAFT_DELAY': nan, 'WEATHER_DELAY': nan}


<h4> Mongo for Departures Aggregation (Avg)

In [6]:
pipeline = [
    {'$match': {'DEPARTURE_DELAY': {'$gt': 0, '$ne': None},
            'DEPARTURE_TIME': {'$ne': None},
            'ARRIVAL_TIME': {'$ne': None}}},
    {'$group': {'_id': '$AIRLINE_x',
                'Average_Delay': {'$avg': '$DEPARTURE_DELAY'}}},
    {'$sort': {'Average_Delay': 1}}
]
result = flights_airlines_collection.aggregate(pipeline)
for e in result:
    print(e)

{'_id': 'Hawaiian Airlines Inc.', 'Average_Delay': 16.754616511968734}
{'_id': 'Alaska Airlines Inc.', 'Average_Delay': 25.991574048974428}
{'_id': 'Southwest Airlines Co.', 'Average_Delay': 26.94361009375152}
{'_id': 'US Airways Inc.', 'Average_Delay': 28.42414096639316}
{'_id': 'Delta Air Lines Inc.', 'Average_Delay': 29.633318581141122}
{'_id': 'Virgin America', 'Average_Delay': 30.120642978003385}
{'_id': 'United Air Lines Inc.', 'Average_Delay': 32.58631860488352}
{'_id': 'American Airlines Inc.', 'Average_Delay': 34.361800390135265}
{'_id': 'JetBlue Airways', 'Average_Delay': 37.59156531292766}
{'_id': 'Skywest Airlines Inc.', 'Average_Delay': 39.241547948118225}
{'_id': 'American Eagle Airlines Inc.', 'Average_Delay': 40.10959796566001}
{'_id': 'Atlantic Southeast Airlines', 'Average_Delay': 40.83302437667995}
{'_id': 'Spirit Air Lines', 'Average_Delay': 41.85487167840109}
{'_id': 'Frontier Airlines Inc.', 'Average_Delay': 44.42139540151773}


<h4> Mongo for Arrivals Aggregation (Avg)

In [7]:
pipeline = [
    {'$match': {'ARRIVAL_DELAY': {'$gt': 0, '$ne': None},
            'ARRIVAL_TIME': {'$ne': None},
            'DIVERTED': 0,
            'CANCELLED': 0}},
    {'$group': {'_id': '$AIRLINE_x', 
                'Average_Arrival_Delay': {'$avg': '$ARRIVAL_DELAY'}}},
    {'$sort': {'Average_Arrival_Delay': 1}}
]
result = flights_airlines_collection.aggregate(pipeline)
for e in result:
    print(e)

{'_id': 'Hawaiian Airlines Inc.', 'Average_Arrival_Delay': 15.341564457202505}
{'_id': 'Alaska Airlines Inc.', 'Average_Arrival_Delay': 22.53564940569906}
{'_id': 'US Airways Inc.', 'Average_Arrival_Delay': 27.389889741440452}
{'_id': 'Southwest Airlines Co.', 'Average_Arrival_Delay': 29.406270854209335}
{'_id': 'Virgin America', 'Average_Arrival_Delay': 30.548130860254677}
{'_id': 'Delta Air Lines Inc.', 'Average_Arrival_Delay': 32.022882190662685}
{'_id': 'Skywest Airlines Inc.', 'Average_Arrival_Delay': 32.43541998936736}
{'_id': 'American Airlines Inc.', 'Average_Arrival_Delay': 34.10820828483717}
{'_id': 'Atlantic Southeast Airlines', 'Average_Arrival_Delay': 35.196197216890596}
{'_id': 'JetBlue Airways', 'Average_Arrival_Delay': 38.204165659884985}
{'_id': 'United Air Lines Inc.', 'Average_Arrival_Delay': 39.156375118341785}
{'_id': 'American Eagle Airlines Inc.', 'Average_Arrival_Delay': 39.446335800871566}
{'_id': 'Spirit Air Lines', 'Average_Arrival_Delay': 40.674984815618224}

<h4> Mongo for Taxi Time Aggregation (Avg)

In [8]:
pipeline = [
    {'$match': {'DEPARTURE_TIME': {'$ne': None},
            'TAXI_OUT': {'$type': 'number'}}},
    {'$group': {'_id': '$AIRLINE_x',
            'AVG_TAXI_TIME': {'$avg': '$TAXI_OUT'}}},
    {'$sort': {'AVG_TAXI_TIME': 1}}]
result = flights_airlines_collection.aggregate(pipeline)
for e in result:
    print(e)

{'_id': 'Southwest Airlines Co.', 'AVG_TAXI_TIME': nan}
{'_id': 'Delta Air Lines Inc.', 'AVG_TAXI_TIME': nan}
{'_id': 'Atlantic Southeast Airlines', 'AVG_TAXI_TIME': nan}
{'_id': 'American Airlines Inc.', 'AVG_TAXI_TIME': nan}
{'_id': 'JetBlue Airways', 'AVG_TAXI_TIME': nan}
{'_id': 'American Eagle Airlines Inc.', 'AVG_TAXI_TIME': nan}
{'_id': 'Frontier Airlines Inc.', 'AVG_TAXI_TIME': nan}
{'_id': 'Alaska Airlines Inc.', 'AVG_TAXI_TIME': nan}
{'_id': 'Hawaiian Airlines Inc.', 'AVG_TAXI_TIME': nan}
{'_id': 'US Airways Inc.', 'AVG_TAXI_TIME': nan}
{'_id': 'United Air Lines Inc.', 'AVG_TAXI_TIME': nan}
{'_id': 'Spirit Air Lines', 'AVG_TAXI_TIME': nan}
{'_id': 'Virgin America', 'AVG_TAXI_TIME': nan}
{'_id': 'Skywest Airlines Inc.', 'AVG_TAXI_TIME': nan}
